# ScientificLoop — GRPO Training Notebook

> **Teaching an LLM to reproduce scientific papers through reinforcement learning.**

This notebook launches GRPO training for **Qwen2.5-Coder-7B-Instruct** with LoRA on HuggingFace's GPU infrastructure via `hf jobs`. Training runs entirely on HF's servers — **no Colab GPU required**. Logs stream into the cell and the job keeps going even if you close the tab.

| Resource | Link |
|----------|------|
| Environment (HF Space) | https://huggingface.co/spaces/Sushant0809/scientific-loop |
| Trained Model | https://huggingface.co/Sushant0809/scientific-loop-grpo |
| Training Script | https://huggingface.co/spaces/Sushant0809/scientific-loop/blob/main/train_grpo.py |
| Blog Post | https://huggingface.co/spaces/Sushant0809/scientific-loop/blob/main/hf_blog_post.md |

---
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sushant0809/scientific-loop/blob/main/train_grpo.ipynb)

## Step 1: Install Dependencies

Only the HF CLI is needed — training runs on HF servers, not this machine.

In [ ]:
%%capture
!pip install -q "huggingface_hub[cli]>=0.23"

# Verify hf CLI is accessible
import subprocess, shutil, sys
hf_path = shutil.which('hf')
if not hf_path:
    # Fallback: find via sys.prefix
    import os
    hf_path = os.path.join(sys.prefix, 'bin', 'hf')

print(f'hf CLI: {hf_path}')
print('Dependencies installed.')

## Step 2: Authenticate with HuggingFace

Add your token: left sidebar → **🔑 Secrets** → **+ Add new secret**  
Name: `HF_TOKEN`  Value: your token from https://huggingface.co/settings/tokens  
Toggle **Notebook access → ON**

In [ ]:
import os
from huggingface_hub import login

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('✓ Logged in to HuggingFace.')
else:
    print('⚠ No HF_TOKEN found. Set it in Colab Secrets (🔑).')
    # HF_TOKEN = 'hf_...'  # ← uncomment and paste as fallback

## Step 3: Launch Training on HuggingFace GPU

This submits `train_grpo.py` to HuggingFace's **H200** (141GB VRAM). Training runs on HF's servers and logs stream into this cell.

**What happens:**
- Installs all ML dependencies on the remote machine
- Loads Qwen2.5-Coder-7B-Instruct
- Runs GRPO with LoRA r=8 for 100 steps (~1.5 hours)
- Pushes trained LoRA adapter + training curves + TensorBoard logs to Hub

**Hardware options** (change `--flavor`):
| Flavor | GPU | VRAM | Cost |
|--------|-----|------|------|
| `h200` | H200 | 141GB | $5/hr |
| `a100-large` | A100 | 80GB | $2.50/hr |
| `a10g-large` | A10G | 24GB | $1.50/hr |

In [ ]:
import subprocess, os, shutil, sys

os.environ['HF_TOKEN'] = HF_TOKEN

# Find hf CLI — handles Colab PATH quirks
hf_cmd = shutil.which('hf') or os.path.join(sys.prefix, 'bin', 'hf')
print(f'Using hf CLI: {hf_cmd}')

# Verify CLI works before launching
check = subprocess.run([hf_cmd, '--version'], capture_output=True, text=True)
print(f'hf version: {check.stdout.strip()}')

result = subprocess.run(
    [
        hf_cmd, 'jobs', 'uv', 'run',
        '--with', 'trl>=1.2.0',
        '--with', 'torch',
        '--with', 'transformers',
        '--with', 'accelerate',
        '--with', 'datasets',
        '--with', 'peft',
        '--with', 'matplotlib',
        '--with', 'tensorboard',
        '--with', 'openenv-scientific-loop @ git+https://huggingface.co/spaces/Sushant0809/scientific-loop',
        '--flavor', 'h200',
        '-s', 'HF_TOKEN',
        '--', 'python', 'train_grpo.py',
    ],
    capture_output=False,
    text=True,
)

if result.returncode == 0:
    print('\n✅ Training completed successfully.')
else:
    print(f'\n❌ Job exited with code {result.returncode}')

## Step 4: View Results

Once training completes, results are automatically pushed to HF Hub.

In [ ]:
from huggingface_hub import HfApi
from IPython.display import Image, display
import requests

MODEL_REPO = 'Sushant0809/scientific-loop-grpo'

# Show training curves
plot_url = f'https://huggingface.co/{MODEL_REPO}/resolve/main/training_curves.png'
resp = requests.get(plot_url)
if resp.status_code == 200:
    display(Image(data=resp.content))
else:
    print('Training curves not yet available — run training first.')

# Show eval scores
try:
    path = HfApi().hf_hub_download(MODEL_REPO, 'eval_scores.jsonl', repo_type='model')
    import json
    print('\nEval Reproduction Scores:')
    with open(path) as f:
        for line in f:
            d = json.loads(line)
            print(f"  Step {d['step']:>4}: {d['score']:.4f}")
except Exception:
    print('Eval scores not yet available.')

print(f'\nFull results: https://huggingface.co/{MODEL_REPO}')
print(f'TensorBoard:  https://huggingface.co/{MODEL_REPO} → Training metrics tab')

## Step 5: Quick Inference Test

Load the trained LoRA adapter and test it on a warmup paper.

In [ ]:
# Install inference dependencies
!pip install -q torch transformers peft \
    "openenv-scientific-loop @ git+https://huggingface.co/spaces/Sushant0809/scientific-loop"

In [ ]:
import torch, re
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from ScientificLoop.paper_corpus import load_paper, format_paper_for_agent, WARMUP_PAPERS
from ScientificLoop.server.execution_engine import run_code, extract_metrics, compute_metric_proximity

BASE_MODEL = 'Qwen/Qwen2.5-Coder-7B-Instruct'
ADAPTER    = 'Sushant0809/scientific-loop-grpo'

SYSTEM_PROMPT = (
    'You are an expert ML engineer. '
    'Read the paper methodology below and implement it as a complete, '
    'self-contained Python script. Output ONLY executable Python code. '
    'The script must print on the last line:\n'
    'METRICS: {"metric_name": value}'
)

print('Loading model + adapter...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16, device_map='auto'
)
model = PeftModel.from_pretrained(base, ADAPTER)
model.eval()
print('Ready.')

def _extract_code(raw):
    fence = re.search(r'```(?:python)?\n(.*?)```', raw.strip(), re.DOTALL)
    return fence.group(1).strip() if fence else raw.strip()

# Test on all warmup papers
for paper in WARMUP_PAPERS:
    prompt = f'{SYSTEM_PROMPT}\n\n{format_paper_for_agent(paper)}'
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=True)
    code = _extract_code(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))
    stdout, stderr, timed_out = run_code(code, paper.execution_timeout)
    achieved = extract_metrics(stdout)
    score, _ = compute_metric_proximity(achieved, paper.target_metrics, paper.metric_weights)
    status = 'TIMEOUT' if timed_out else ('ERROR' if stderr and not stdout else 'OK')
    print(f'[{paper.paper_id}]  status={status}  score={score:.3f}  achieved={achieved}')

## Links

- **Environment (HF Space):** https://huggingface.co/spaces/Sushant0809/scientific-loop
- **Trained Model:** https://huggingface.co/Sushant0809/scientific-loop-grpo
- **Training Script:** https://huggingface.co/spaces/Sushant0809/scientific-loop/blob/main/train_grpo.py
- **Blog Post:** https://huggingface.co/spaces/Sushant0809/scientific-loop/blob/main/hf_blog_post.md
- **GitHub:** https://github.com/Sushant0809/scientific-loop

---
*OpenEnv Hackathon — PyTorch × Scaler — April 2026*